# Create DNRF Awards (Danmarks Grundforskningsfond / Danish National Research Foundation)

Creates DNRF awards from the foundation's own website (dg.dk, WordPress). DNRF publishes no
bulk export or grants API, so `scripts/local/dnrf_to_s3.py` combines:
- the WP REST API (`/wp-json/wp/v2/centers`) + English detail pages for every **active** grant
  (Centers of Excellence, DNRF Chairs, Pioneer Centres, Niels Bohr Professorships): leader,
  period, application round, host institution, grant amount (DKK);
- `/en/former-centers-of-excellence/` for every **former** Center of Excellence (name, head of
  center, host, establishment year; no amounts/dates published);
- the retired-programme pages (Niels Bohr Professorships 2013-2022, Niels Bohr Visiting
  Professorships 2006-2011, DG Professorships 2007-2012): name, host, grant period.

**174 grants, 1993-2025** (local run 2026-09-30): 140 Centers of Excellence, 12 Niels Bohr
Professorships, 9 DNRF Chairs, 6 Niels Bohr Visiting Professorships, 4 Pioneer Centres,
3 DG Professorships. 100% title/PI, ~98% host institution, ~27% amount (DKK 3.49B; dg.dk only
publishes amounts for active grants), start year ~95%.

**Amount / currency:** DKK as printed on the detail page ("54,6 M DKK", "60.000.000 DKK",
"DKK 250 million"). Pioneer Centre amounts are the published grant total; Pioneer Centres are
co-funded with other Danish foundations, so treat those four as the programme total. Amount
coverage is low because DNRF does not publish amounts for former grants (not a mapping gap).

**`funder_award_id` (runbook 2.1.1):** citing works write `DNRF<n>` ("CENPERM DNRF100",
"HADAL, Grant No. DNRF145"), but dg.dk never prints the number. `scripts/local/dnrf_grant_numbers.csv`
is a static crosswalk built from acknowledgement strings already in `openalex_awards_raw`
for F4320320943 that name both the center and the number (consensus + uniqueness required):
**28 grants ship as `DNRF<n>`** and collapse onto ~100 existing work-funder shells with that
exact string. The other 146 ship a stable synthetic key `DG-<programme>-<key>`
(`DG-COE-61862` = REST post id; `DG-COE-former-2012-cenperm` = round year + acronym/title slug;
`DG-NBP-<name>` for retired professorships). The `dnrf_number` raw column marks crosswalk rows.

**Prerequisites:** run `scripts/local/dnrf_to_s3.py` (uploads
`s3://openalex-ingest/awards/dnrf/dnrf_projects.parquet`, 1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320320943`, Danmarks
Grundforskningsfond, ror 00znyv691, doi 10.13039/501100001732 (read from `openalex.funders.funders`).

**Priority:** `484` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.dnrf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/dnrf/dnrf_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_grants, COUNT(DISTINCT funder_award_id) as distinct_ids,
       COUNT(dnrf_number) as with_dnrf_number
FROM openalex.awards.dnrf_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.dnrf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.dnrf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320320943 is Crossref-registered, so it MUST resolve to exactly 1 row in `openalex.funders.funders`.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320320943;

## Step 2: Create DNRF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.dnrf_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320320943  -- Danmarks Grundforskningsfond
),

g AS (
    SELECT
        r.*,
        TRY_TO_DATE(r.start_date, 'yyyy-MM-dd') AS sd,
        TRY_TO_DATE(r.end_date, 'yyyy-MM-dd') AS ed,
        -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation
        CASE WHEN NULLIF(TRIM(r.lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(r.lead_given_name), ''),
            'family_name', NULLIF(TRIM(r.lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(r.host_institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead,
        CASE WHEN NULLIF(TRIM(r.co_lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(r.co_lead_given_name), ''),
            'family_name', NULLIF(TRIM(r.co_lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(r.host_institution), ''),
                'country', CAST(NULL AS STRING),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS co_lead
    FROM openalex.awards.dnrf_raw r
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    TRY_CAST(g.amount AS DOUBLE) as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) IS NOT NULL THEN 'DKK' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE WHEN g.programme IN ('Center of Excellence', 'Pioneer Centre') THEN 'research'
         ELSE 'fellowship' END as funding_type,
    g.programme as funder_scheme,
    'dnrf_dg_dk' as provenance,
    g.sd as start_date,
    g.ed as end_date,
    -- former centers publish only the round's establishment year; Pioneer Centres a year range
    COALESCE(YEAR(g.sd), TRY_CAST(g.start_year AS INT), TRY_CAST(g.established_year AS INT)) as start_year,
    COALESCE(YEAR(g.ed), TRY_CAST(g.end_year AS INT)) as end_year,
    g.lead as lead_investigator,
    g.co_lead as co_lead_investigator,
    CASE WHEN g.lead IS NOT NULL AND g.co_lead IS NOT NULL THEN array(g.lead, g.co_lead)
         WHEN g.lead IS NOT NULL THEN array(g.lead) END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM g
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'dnrf_dg_dk' AND priority = 484;

-- Priority 484: direct-from-funder ingest of DNRF's own website. Higher wins under the
-- 2026-06-20 DESC dedup (oxjob #500), so the 28 DNRF<n> rows outrank the priority-0
-- work-funder shells that share their award id.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    484 as priority
FROM openalex.awards.dnrf_awards;

## Verification Queries

In [ ]:
%sql
-- Shape check (2.1.1): every id is DNRF<n> (crosswalk) or the synthetic DG-<programme>-<key>.
SELECT
    COUNT(*) as total,
    COUNT(DISTINCT id) as distinct_ids,
    COUNT_IF(funder_award_id RLIKE '^DNRF[0-9]{2,3}$') as dnrf_number_ids,
    COUNT_IF(funder_award_id RLIKE '^DG-(COE|CHAIR|PIONEER|NBP|NBVP|DGP)-.+$') as synthetic_ids,
    COUNT_IF(NOT (funder_award_id RLIKE '^DNRF[0-9]{2,3}$' OR funder_award_id RLIKE '^DG-(COE|CHAIR|PIONEER|NBP|NBVP|DGP)-.+$')) as bad_shape
FROM openalex.awards.dnrf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.dnrf_awards ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, COUNT(amount) as with_amount, ROUND(SUM(amount), 0) as total_dkk,
       MIN(start_year) as first_year, MAX(start_year) as last_year
FROM openalex.awards.dnrf_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt FROM openalex.awards.dnrf_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- 6.4a frequency check: PI top-20 must be a real long tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.dnrf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    COUNT(start_date) as has_start_date,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.dnrf_awards;

In [ ]:
%sql
-- Overlap with existing work-funder shells: the DNRF<n> rows should share ids with shells.
SELECT
    COUNT(DISTINCT d.id) as dnrf_number_awards,
    COUNT(DISTINCT CASE WHEN s.id IS NOT NULL THEN d.id END) as with_matching_shell,
    COUNT(s.id) as shell_rows_upgraded
FROM openalex.awards.dnrf_awards d
LEFT JOIN openalex.awards.openalex_awards_raw s
  ON s.id = d.id AND s.priority = 0
WHERE d.funder_award_id RLIKE '^DNRF[0-9]+$';

In [ ]:
%sql
-- 6.8: rows reached the shared raw table at the assigned priority.
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'dnrf_dg_dk'
GROUP BY provenance, priority;